# 🧠 LLaMAC PPG 기반 실시간 감정 예측 모델 (GPU 최적화 버전)

## PPG 단일 센서를 이용한 연속형 감정 변수 비선형 회귀 예측

---

이 노트북은 **LLaMAC (Low-cost Biosignal Sensor based Large Multimodal Dataset for Affective Computing)** 데이터셋에서 **PPG(Photoplethysmography) 센서 데이터만을 활용**하여 5가지 연속형 감정 변수를 비선형 회귀 방식으로 예측하는 완전한 파이프라인을 제공합니다.

### 🚀 하이엔드 환경 (5090 GPU + 64GB RAM) 최적화
- **전체 데이터(108명) 일괄 로딩**을 지원하는 넉넉한 메모리 관리
- XGBoost, LightGBM, CatBoost의 **GPU 가속** 지원
- SVR 포함 6개 모델 전체 비교 평가

### 예측 대상 변수
| 변수 | 스케일 | 설명 |
|------|--------|------|
| Valence | 1~5 | 감정의 긍정/부정 정도 |
| Arousal | 1~5 | 감정의 각성/흥분 정도 |
| Dominance | 1~5 | 감정의 지배/통제 정도 |
| Liking | 1~3 | 자극에 대한 선호도 |
| Emotion Intensity | 1~6 | 감정 강도 |


## 1. 환경 설정 및 Configuration

필요한 라이브러리를 설치하고, 사용자가 쉽게 조절할 수 있는 **Configuration**을 정의합니다.
데이터 크기, 윈도우 크기, Optuna 튜닝 범위, GPU 사용 여부 등을 여기서 모두 제어할 수 있습니다.

In [ ]:
# ============================================================
# 필수 라이브러리 설치 (필요시 주석 해제)
# ============================================================
# !pip install -q neurokit2 optuna xgboost lightgbm catboost scikit-learn

import os, sys, glob, json, time, gc, zipfile
import urllib.request
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.signal as signal
from scipy.stats import skew, kurtosis
from scipy.interpolate import interp1d

from sklearn.model_selection import train_test_split, KFold, cross_val_score, learning_curve
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

def _env_int(name, default):
    value = os.environ.get(name)
    return default if value in (None, "") else int(value)

def _env_bool(name, default):
    value = os.environ.get(name)
    if value in (None, ""):
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}

SMOKE_MODE = _env_bool("LLAMAC_SMOKE", False)

# 시각화 설정
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

# ============================================================
# ⚙️ Configuration (사용자 설정)
# ============================================================
CONFIG = {
    # 1. 데이터 설정
    "max_participants": _env_int("LLAMAC_MAX_PARTICIPANTS", 6 if SMOKE_MODE else 108),  # 분석할 참가자 수
    "window_sec": 30,         # 피처 추출 타임윈도우 크기 (초)
    "step_sec": _env_int("LLAMAC_STEP_SEC", 30 if SMOKE_MODE else 5),  # Rolling window 스텝 크기 (초)
    "sampling_rate": 64,      # 리샘플링 목표 주파수 (Hz)
    "raw_dir": os.environ.get("LLAMAC_RAW_DIR", "data/raw"),
    "extract_dir": os.environ.get("LLAMAC_EXTRACT_DIR", "data/extracted"),
    "processed_dir": os.environ.get("LLAMAC_PROCESSED_DIR", "data/processed"),
    "verify_md5": _env_bool("LLAMAC_VERIFY_MD5", True),
    "force_download": _env_bool("LLAMAC_FORCE_DOWNLOAD", False),
    "force_extract": _env_bool("LLAMAC_FORCE_EXTRACT", False),
    
    # 2. 모델 평가 및 학습 설정
    "cv_folds": _env_int("LLAMAC_CV_FOLDS", 2 if SMOKE_MODE else 5),  # Cross-validation 분할 수
    "use_gpu": _env_bool("LLAMAC_USE_GPU", False),  # GPU 사용 여부 (XGBoost, LightGBM, CatBoost)
    "random_state": 42,       # 재현성을 위한 시드
    
    # 3. Optuna 튜닝 설정
    "n_trials": _env_int("LLAMAC_N_TRIALS", 2 if SMOKE_MODE else 50),  # Optuna 탐색 횟수
    "optuna_ranges": {
        "Random Forest": {
            "n_estimators": (50, 500),
            "max_depth": (5, 25),
            "min_samples_split": (2, 15),
            "min_samples_leaf": (1, 10)
        },
        "Extra Trees": {
            "n_estimators": (50, 500),
            "max_depth": (5, 25),
            "min_samples_split": (2, 15),
            "min_samples_leaf": (1, 10)
        },
        "XGBoost": {
            "n_estimators": (50, 500),
            "learning_rate": (0.005, 0.3),
            "max_depth": (3, 12),
            "subsample": (0.5, 1.0),
            "colsample_bytree": (0.5, 1.0),
            "reg_alpha": (1e-5, 5.0),
            "reg_lambda": (1e-5, 5.0)
        },
        "LightGBM": {
            "n_estimators": (50, 500),
            "learning_rate": (0.005, 0.3),
            "num_leaves": (15, 128),
            "max_depth": (3, 15),
            "subsample": (0.5, 1.0),
            "colsample_bytree": (0.5, 1.0),
            "reg_alpha": (1e-5, 5.0),
            "reg_lambda": (1e-5, 5.0)
        },
        "CatBoost": {
            "iterations": (50, 500),
            "learning_rate": (0.005, 0.3),
            "depth": (4, 10),
            "l2_leaf_reg": (1e-5, 5.0)
        },
        "SVR": {
            "C": (0.1, 100.0),
            "epsilon": (0.01, 1.0),
            "gamma": (0.001, 1.0)
        }
    }
}

if SMOKE_MODE:
    for ranges in CONFIG["optuna_ranges"].values():
        if "n_estimators" in ranges:
            ranges["n_estimators"] = (10, 20)
        if "iterations" in ranges:
            ranges["iterations"] = (10, 20)

np.random.seed(CONFIG["random_state"])

display(HTML(f'''
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white;">
<h3>✅ 환경 설정 및 Configuration 완료</h3>
<ul>
<li><strong>참가자 수</strong>: {CONFIG["max_participants"]}명</li>
<li><strong>윈도우/스텝</strong>: {CONFIG["window_sec"]}초 / {CONFIG["step_sec"]}초</li>
<li><strong>데이터 경로</strong>: {CONFIG["extract_dir"]}</li>
<li><strong>GPU 사용</strong>: {'활성화 🚀' if CONFIG["use_gpu"] else '비활성화'}</li>
<li><strong>Optuna Trials</strong>: {CONFIG["n_trials"]}회</li>
<li><strong>Smoke Mode</strong>: {'ON' if SMOKE_MODE else 'OFF'}</li>
</ul>
</div>
'''))


## 2. LLaMAC 데이터셋 다운로드

Figshare API를 통해 LLaMAC 데이터셋을 다운로드합니다. Configuration의 `max_participants` 설정에 따라 다운로드 수가 결정됩니다.

In [ ]:
# ============================================================
# LLaMAC 데이터셋 준비: raw 다운로드, extracted 검증, processed index 생성
# ============================================================
FIGSHARE_ARTICLE_ID = "28748696"
FIGSHARE_VERSION = "6"
API_URL = f"https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE_ID}/versions/{FIGSHARE_VERSION}"
USER_AGENT = "llamac-research-notebook/1.0"

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path.cwd().resolve()


def resolve_project_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else PROJECT_ROOT / path


RAW_DIR = resolve_project_path(CONFIG["raw_dir"])
EXTRACT_DIR = resolve_project_path(CONFIG["extract_dir"])
PROCESSED_DIR = resolve_project_path(CONFIG["processed_dir"])
MANIFEST_PATH = RAW_DIR / "llamac_figshare_manifest.json"
DATASET_INDEX_PATH = PROCESSED_DIR / "dataset_index.csv"

for directory in [RAW_DIR, EXTRACT_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def natural_key(text):
    import re
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", str(text))]


def safe_name(name):
    name = name.replace("\\", "/").split("/")[-1]
    if not name or name in {".", ".."}:
        raise ValueError(f"Unsafe Figshare filename: {name!r}")
    return name


def md5sum(path, chunk_size=1024 * 1024):
    import hashlib
    digest = hashlib.md5()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def raw_file_complete(path, expected_size=None, expected_md5=None):
    path = Path(path)
    if not path.exists() or not path.is_file():
        return False
    if expected_size is not None and path.stat().st_size != int(expected_size):
        return False
    if CONFIG["verify_md5"] and expected_md5 and md5sum(path) != expected_md5:
        return False
    return True


def participant_data_dir(pid):
    root = EXTRACT_DIR / str(pid)
    direct = root / "answer.csv"
    if direct.exists():
        return root
    nested = root / str(pid) / "answer.csv"
    if nested.exists():
        return root / str(pid)
    matches = sorted(root.glob("*/answer.csv"), key=lambda p: natural_key(p.parent.name)) if root.exists() else []
    return matches[0].parent if matches else root


def valid_participant_dir(pid):
    directory = participant_data_dir(pid)
    return directory.exists() and (directory / "answer.csv").exists() and any(directory.glob("band_*.csv"))


def discovered_participants():
    ids = []
    for path in sorted(EXTRACT_DIR.iterdir(), key=lambda p: natural_key(p.name)) if EXTRACT_DIR.exists() else []:
        if path.is_dir() and path.name.isdigit() and valid_participant_dir(path.name):
            ids.append(path.name)
    return ids


def fetch_metadata():
    request = urllib.request.Request(API_URL, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.loads(response.read().decode())


def load_or_fetch_manifest(required_participants):
    existing = discovered_participants()
    if len(existing) >= required_participants and MANIFEST_PATH.exists():
        print(f"기존 추출 데이터 확인 완료: {len(existing)}명. Figshare API 조회/다운로드 생략")
        return json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if len(existing) >= required_participants and not MANIFEST_PATH.exists():
        print(f"기존 추출 데이터 확인 완료: {len(existing)}명. manifest 없이 다운로드 생략")
        return None
    if MANIFEST_PATH.exists():
        print(f"기존 manifest 사용: {MANIFEST_PATH}")
        return json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

    print("로컬 데이터가 부족해 Figshare metadata를 조회합니다.")
    metadata = fetch_metadata()
    files = [item for item in metadata.get("files", []) if item.get("download_url")]
    manifest = {
        "title": metadata.get("title"),
        "doi": metadata.get("doi"),
        "article_id": metadata.get("id"),
        "version": metadata.get("version"),
        "published_date": metadata.get("published_date"),
        "license": metadata.get("license"),
        "source_api": API_URL,
        "file_count": len(files),
        "total_size_bytes": sum(int(f.get("size", 0)) for f in files),
        "files": files,
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"manifest 저장: {MANIFEST_PATH}")
    return manifest


def participant_files_from_manifest(manifest, max_participants):
    if manifest is None:
        return []
    files = []
    for item in manifest.get("files", []):
        name = safe_name(str(item["name"]))
        if name.endswith(".zip") and name[:-4].isdigit():
            copied = dict(item)
            copied["name"] = name
            copied["md5"] = item.get("md5") or item.get("computed_md5") or item.get("supplied_md5")
            files.append(copied)
    files.sort(key=lambda f: natural_key(f["name"]))
    return files[:max_participants]


def safe_zip_members(zf):
    members = []
    for info in zf.infolist():
        member = Path(info.filename)
        if member.is_absolute() or ".." in member.parts:
            raise ValueError(f"Unsafe zip member path: {info.filename}")
        members.append(info)
    return members


def download_raw_file(file_info):
    target = RAW_DIR / safe_name(file_info["name"])
    expected_size = file_info.get("size")
    expected_md5 = file_info.get("md5")
    if not CONFIG["force_download"] and raw_file_complete(target, expected_size, expected_md5):
        return "raw skipped", target

    temp = target.with_suffix(target.suffix + ".part")
    request = urllib.request.Request(file_info["download_url"], headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(request, timeout=120) as response, temp.open("wb") as out:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            out.write(chunk)
    if not raw_file_complete(temp, expected_size, expected_md5):
        temp.unlink(missing_ok=True)
        raise RuntimeError(f"다운로드 검증 실패: {file_info['name']}")
    temp.replace(target)
    return "downloaded", target


def extract_participant_archive(file_info):
    pid = safe_name(file_info["name"]).replace(".zip", "")
    if valid_participant_dir(pid) and not CONFIG["force_extract"]:
        return "extract skipped", participant_data_dir(pid)

    _, archive = download_raw_file(file_info)
    target = EXTRACT_DIR / pid
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        safe_zip_members(zf)
        zf.extractall(target)
    marker = target / ".extracted_ok"
    marker.write_text(time.strftime("%Y-%m-%d %H:%M:%S %Z"), encoding="utf-8")
    if not valid_participant_dir(pid):
        raise RuntimeError(f"압축 해제 후 필수 파일을 찾지 못했습니다: {target}")
    return "extracted", participant_data_dir(pid)


def build_dataset_index():
    rows = []
    for pid in discovered_participants():
        directory = participant_data_dir(pid)
        for path in sorted(directory.rglob("*"), key=lambda p: natural_key(str(p.relative_to(directory)))):
            if not path.is_file() or path.name == ".extracted_ok":
                continue
            rows.append({
                "participant_id": pid,
                "file_name": path.name,
                "relative_path": str(path.relative_to(EXTRACT_DIR)),
                "suffix": path.suffix.lower(),
                "size_bytes": path.stat().st_size,
            })
    pd.DataFrame(rows, columns=["participant_id", "file_name", "relative_path", "suffix", "size_bytes"]).to_csv(DATASET_INDEX_PATH, index=False)
    return DATASET_INDEX_PATH, len(rows)


def prepare_dataset(max_participants):
    before = discovered_participants()
    manifest = load_or_fetch_manifest(max_participants)
    files = participant_files_from_manifest(manifest, max_participants)

    if len(before) < max_participants:
        if not files:
            raise RuntimeError(
                f"추출 데이터가 {len(before)}명뿐이고 manifest/file metadata가 없어 부족분을 받을 수 없습니다. "
                f"{MANIFEST_PATH} 또는 data/raw/*.zip을 확인하세요."
            )
        for file_info in files:
            pid = file_info["name"].replace(".zip", "")
            if valid_participant_dir(pid) and not CONFIG["force_extract"]:
                continue
            status, path = extract_participant_archive(file_info)
            print(f"[{status}] {pid} -> {path}")

    available = discovered_participants()
    if len(available) < max_participants:
        raise RuntimeError(f"필요 참가자 {max_participants}명 중 {len(available)}명만 준비되었습니다: {EXTRACT_DIR}")

    index_path, indexed_files = build_dataset_index()
    print(f"데이터 준비 완료: participants={len(available)}, raw={RAW_DIR}, extracted={EXTRACT_DIR}")
    print(f"검증 index: {index_path} ({indexed_files} files)")
    return [participant_data_dir(pid) for pid in available[:max_participants]]


PARTICIPANT_DIRS = prepare_dataset(CONFIG["max_participants"])
DATA_DIR = EXTRACT_DIR


## 3. 데이터 로딩 및 탐색

각 참가자의 `answer.csv`에서 라벨을, `band_#.csv`에서 PPG 신호를 추출합니다.
64GB RAM 환경이므로 메모리 제약 없이 전체 데이터를 리스트에 보관합니다.

In [ ]:
# ============================================================
# 데이터 로딩
# ============================================================
participant_dirs = PARTICIPANT_DIRS[:CONFIG["max_participants"]]
print(f"탐색된 참가자 수: {len(participant_dirs)}")

all_data = []  # (ppg_resampled_array, label_dict)
WINDOW_SAMPLES = CONFIG["sampling_rate"] * CONFIG["window_sec"]

for pdir in participant_dirs:
    pdir = Path(pdir)
    pid = pdir.name
    answer_path = pdir / 'answer.csv'
    if not answer_path.exists():
        continue

    labels_df = pd.read_csv(answer_path)
    band_files = sorted(pdir.glob('band_*.csv'), key=lambda p: natural_key(p.name))

    for bf in band_files:
        try:
            trial_num = int(bf.stem.replace('band_', ''))
            if trial_num > len(labels_df):
                continue

            df = pd.read_csv(bf)
            ppg_raw = df['PPG'].to_numpy(dtype=float)
            ppg_time = df['PPG_Time'].to_numpy(dtype=float)

            # NaN 제거
            valid = ~(np.isnan(ppg_raw) | np.isnan(ppg_time))
            ppg_raw, ppg_time = ppg_raw[valid], ppg_time[valid]
            if len(ppg_raw) < 100:
                continue

            # 리샘플링 (균일 간격)
            duration = ppg_time[-1] - ppg_time[0]
            n_target = int(duration * CONFIG["sampling_rate"])
            if n_target < WINDOW_SAMPLES:
                continue

            t_uniform = np.linspace(ppg_time[0], ppg_time[-1], n_target)
            f_interp = interp1d(ppg_time, ppg_raw, kind='linear', fill_value='extrapolate')
            ppg_resampled = f_interp(t_uniform)

            label_row = labels_df.iloc[trial_num - 1]
            label = {
                'Valence': float(label_row['Valence']),
                'Arousal': float(label_row['Arousal']),
                'Dominance': float(label_row['Dominance']),
                'Liking': float(label_row['Liking']),
                'Emotion_Intensity': float(label_row['EmotStr']),
                'participant_id': pid,
                'trial_num': trial_num,
            }
            all_data.append((ppg_resampled, label))
        except Exception as e:
            print(f"스킵: {bf} ({type(e).__name__}: {e})")
            continue

if len(all_data) == 0:
    raise RuntimeError(f"사용 가능한 PPG trial을 찾지 못했습니다. 데이터 경로를 확인하세요: {DATA_DIR}")
print(f"로드된 전체 트라이얼 수: {len(all_data)}")


## 4. PPG 피처 엔지니어링

실시간 바이오피드백 장치에서 단위 rolling 예측을 위해, Configuration에 정의된 윈도우 크기에서 다양한 피처를 추출합니다.

### 추출 피처 카테고리 (총 36개)
| 카테고리 | 피처 | 설명 |
|----------|------|------|
| 심박수 (6) | hr_mean, hr_std, hr_min, hr_max, hr_range, hr_median | PPG 피크 간격 기반 |
| 시간도메인 HRV (7) | mean_nn, sdnn, rmssd, sdsd, pnn50, pnn20, cvnn | IBI(Inter-Beat Interval) 기반 |
| 주파수도메인 HRV (5) | lf, hf, lfhf, lf_norm, hf_norm | Welch PSD 기반 |
| 비선형 HRV (3) | sd1, sd2, sd1sd2 | Poincaré Plot 기반 |
| 형태학적 (6) | amp_mean, amp_std, peak_amp, peak_std, pp_mean, pp_std | PPG 파형 형태 |
| 통계적 (9) | mean, std, skew, kurt, iqr, zcr, energy, diff_abs, diff_std | 신호 통계량 |

In [ ]:
# ============================================================
# PPG 피처 추출 함수
# ============================================================

def bandpass_filter(sig, lowcut=0.5, highcut=8.0, fs=64, order=3):
    nyq = 0.5 * fs
    b, a = signal.butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return signal.filtfilt(b, a, sig)

def find_ppg_peaks(ppg_cleaned, fs=64):
    min_dist = int(fs * 0.4)
    peaks, _ = signal.find_peaks(ppg_cleaned, distance=min_dist, 
                                  height=np.percentile(ppg_cleaned, 30))
    return peaks

def extract_ppg_features(ppg_window, fs=64):
    features = {}
    
    # 전처리: 대역통과 필터 + Z-score 정규화
    ppg_filtered = bandpass_filter(ppg_window, fs=fs)
    ppg_z = (ppg_filtered - np.mean(ppg_filtered)) / (np.std(ppg_filtered) + 1e-8)
    peaks = find_ppg_peaks(ppg_z, fs)
    
    # === 1. 심박수 피처 ===
    if len(peaks) >= 3:
        hr_inst = 60.0 / (np.diff(peaks) / fs)
        hr_inst = hr_inst[(hr_inst > 30) & (hr_inst < 200)]
        if len(hr_inst) > 0:
            features['hr_mean'] = np.mean(hr_inst)
            features['hr_std'] = np.std(hr_inst)
            features['hr_min'] = np.min(hr_inst)
            features['hr_max'] = np.max(hr_inst)
            features['hr_range'] = features['hr_max'] - features['hr_min']
            features['hr_median'] = np.median(hr_inst)
        else:
            for k in ['hr_mean','hr_std','hr_min','hr_max','hr_range','hr_median']:
                features[k] = np.nan
    else:
        for k in ['hr_mean','hr_std','hr_min','hr_max','hr_range','hr_median']:
            features[k] = np.nan
    
    # === 2. 시간도메인 HRV ===
    if len(peaks) >= 3:
        ibi = np.diff(peaks) / fs * 1000  # ms
        ibi = ibi[(ibi > 300) & (ibi < 2000)]
    else:
        ibi = np.array([])
    
    if len(ibi) >= 3:
        features['hrv_mean_nn'] = np.mean(ibi)
        features['hrv_sdnn'] = np.std(ibi, ddof=1)
        diff_ibi = np.diff(ibi)
        features['hrv_rmssd'] = np.sqrt(np.mean(diff_ibi**2))
        features['hrv_sdsd'] = np.std(diff_ibi, ddof=1) if len(diff_ibi) > 1 else 0
        features['hrv_pnn50'] = np.sum(np.abs(diff_ibi) > 50) / max(len(diff_ibi), 1) * 100
        features['hrv_pnn20'] = np.sum(np.abs(diff_ibi) > 20) / max(len(diff_ibi), 1) * 100
        features['hrv_cvnn'] = (features['hrv_sdnn'] / features['hrv_mean_nn']) * 100
    else:
        for k in ['hrv_mean_nn','hrv_sdnn','hrv_rmssd','hrv_sdsd','hrv_pnn50','hrv_pnn20','hrv_cvnn']:
            features[k] = np.nan
    
    # === 3. 주파수도메인 HRV ===
    if len(ibi) >= 5:
        try:
            ibi_t = np.cumsum(ibi) / 1000.0
            ibi_t -= ibi_t[0]
            t_uni = np.arange(0, ibi_t[-1], 0.25)
            if len(t_uni) > 8:
                fi = interp1d(ibi_t, ibi[:len(ibi_t)], kind='linear', fill_value='extrapolate')
                ibi_uni = fi(t_uni)
                freqs, psd = signal.welch(ibi_uni - np.mean(ibi_uni), fs=4.0, 
                                         nperseg=min(len(ibi_uni), 32))
                lf_mask = (freqs >= 0.04) & (freqs < 0.15)
                hf_mask = (freqs >= 0.15) & (freqs < 0.4)
                features['hrv_lf'] = np.trapz(psd[lf_mask], freqs[lf_mask])
                features['hrv_hf'] = np.trapz(psd[hf_mask], freqs[hf_mask])
                features['hrv_lfhf'] = features['hrv_lf'] / max(features['hrv_hf'], 1e-6)
                tp = features['hrv_lf'] + features['hrv_hf']
                features['hrv_lf_norm'] = features['hrv_lf'] / max(tp, 1e-6) * 100
                features['hrv_hf_norm'] = features['hrv_hf'] / max(tp, 1e-6) * 100
            else:
                raise ValueError()
        except:
            for k in ['hrv_lf','hrv_hf','hrv_lfhf','hrv_lf_norm','hrv_hf_norm']:
                features[k] = np.nan
    else:
        for k in ['hrv_lf','hrv_hf','hrv_lfhf','hrv_lf_norm','hrv_hf_norm']:
            features[k] = np.nan
    
    # === 4. 비선형 HRV ===
    if len(ibi) >= 4:
        d = np.diff(ibi)
        features['hrv_sd1'] = np.std(d, ddof=1) / np.sqrt(2)
        sd2sq = 2 * np.var(ibi, ddof=1) - features['hrv_sd1']**2
        features['hrv_sd2'] = np.sqrt(max(sd2sq, 0))
        features['hrv_sd1sd2'] = features['hrv_sd1'] / max(features['hrv_sd2'], 1e-6)
    else:
        for k in ['hrv_sd1','hrv_sd2','hrv_sd1sd2']:
            features[k] = np.nan
    
    # === 5. 형태학적 피처 ===
    features['ppg_amp_mean'] = np.mean(np.abs(ppg_z))
    features['ppg_amp_std'] = np.std(np.abs(ppg_z))
    if len(peaks) >= 2:
        features['ppg_peak_amp'] = np.mean(ppg_z[peaks])
        features['ppg_peak_std'] = np.std(ppg_z[peaks])
        features['ppg_pp_mean'] = np.mean(np.diff(peaks) / fs)
        features['ppg_pp_std'] = np.std(np.diff(peaks) / fs)
    else:
        for k in ['ppg_peak_amp','ppg_peak_std','ppg_pp_mean','ppg_pp_std']:
            features[k] = np.nan
    
    # === 6. 통계적 피처 ===
    features['ppg_mean'] = np.mean(ppg_filtered)
    features['ppg_std'] = np.std(ppg_filtered)
    features['ppg_skew'] = skew(ppg_filtered)
    features['ppg_kurt'] = kurtosis(ppg_filtered)
    features['ppg_iqr'] = np.percentile(ppg_filtered, 75) - np.percentile(ppg_filtered, 25)
    features['ppg_zcr'] = np.sum(np.diff(np.sign(ppg_z)) != 0) / len(ppg_z)
    features['ppg_energy'] = np.sum(ppg_filtered**2) / len(ppg_filtered)
    d = np.diff(ppg_filtered)
    features['ppg_diff_abs'] = np.mean(np.abs(d))
    features['ppg_diff_std'] = np.std(d)
    
    return features


## 5. 전체 데이터셋 피처 추출

각 트라이얼에서 Configuration에 설정된 윈도우 크기와 스텝 간격으로 슬라이딩하며 피처를 추출합니다.

In [ ]:
# ============================================================
# 전체 데이터셋에서 피처 추출 (Rolling Window)
# ============================================================
print(f"{CONFIG['window_sec']}초 윈도우, {CONFIG['step_sec']}초 스텝으로 피처 추출 중...")
start_time = time.time()

feature_rows = []
step_samples = CONFIG["step_sec"] * CONFIG["sampling_rate"]

for ppg_arr, label in all_data:
    n = len(ppg_arr)
    start = 0
    # 전체 데이터 사용 (메모리 제약이 없으므로 트라이얼 내 가능한 모든 윈도우 추출)
    while start + WINDOW_SAMPLES <= n:
        window = ppg_arr[start:start + WINDOW_SAMPLES]
        feats = extract_ppg_features(window, CONFIG["sampling_rate"])
        if feats:
            feats.update(label)
            feature_rows.append(feats)
        start += step_samples

feature_df = pd.DataFrame(feature_rows)
if feature_df.empty:
    raise RuntimeError("피처 추출 결과가 비어 있습니다. window_sec/step_sec 또는 원본 PPG 길이를 확인하세요.")
elapsed = time.time() - start_time

TARGET_COLS = ['Valence', 'Arousal', 'Dominance', 'Liking', 'Emotion_Intensity']
META_COLS = ['participant_id', 'trial_num']
FEATURE_COLS = [c for c in feature_df.columns if c not in TARGET_COLS + META_COLS]

print(f"\n✅ 피처 추출 완료! (소요시간: {elapsed:.1f}초)")
print(f"   총 샘플 수: {len(feature_df):,}")
print(f"   피처 수: {len(FEATURE_COLS)}")


## 6. 데이터 전처리 및 탐색적 분석

NaN 처리, 이상치 클리핑, 스케일링을 수행합니다.

In [ ]:
# ============================================================
# 데이터 전처리
# ============================================================

# NaN 처리 (중앙값 대체)
feature_df[FEATURE_COLS] = feature_df[FEATURE_COLS].fillna(feature_df[FEATURE_COLS].median())
feature_df[FEATURE_COLS] = feature_df[FEATURE_COLS].replace([np.inf, -np.inf], np.nan).fillna(0)

# 이상치 클리핑 (1~99 percentile)
for col in FEATURE_COLS:
    Q1, Q3 = feature_df[col].quantile(0.01), feature_df[col].quantile(0.99)
    feature_df[col] = feature_df[col].clip(Q1, Q3)

# 타겟 분포 시각화
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, t in enumerate(TARGET_COLS):
    axes[i].hist(feature_df[t], bins=20, color=plt.cm.Set2(i), edgecolor='white', alpha=0.8)
    axes[i].set_title(t, fontweight='bold')
    axes[i].set_xlabel('Value')
plt.suptitle('타겟 변수 분포', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. 피처 선택 (Feature Importance 기반)

Random Forest의 Feature Importance를 활용하여 기여도가 낮은 하위 20% 피처를 제거합니다.

In [ ]:
# ============================================================
# Feature Importance 기반 피처 선택
# ============================================================
X = feature_df[FEATURE_COLS].values
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

rf_imp = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=CONFIG["random_state"], n_jobs=-1)
rf_imp.fit(X_scaled, feature_df['Arousal'].values)
importances = pd.Series(rf_imp.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

# 하위 20% 제거
threshold = importances.quantile(0.2)
SELECTED = importances[importances >= threshold].index.tolist()
REMOVED = importances[importances < threshold].index.tolist()

print(f"원래 피처 수: {len(FEATURE_COLS)}")
print(f"선택된 피처 수: {len(SELECTED)}")
print(f"제거된 피처: {REMOVED}")

## 8. 6개 비선형 회귀 모델 비교 평가

Configuration의 `use_gpu` 설정에 따라 GPU 가속을 적용하여 6개 모델(RF, XGBoost, LightGBM, CatBoost, Extra Trees, SVR)을 비교합니다.

In [ ]:
# ============================================================
# 6개 모델 비교 평가 (SVR 포함, GPU 지원)
# ============================================================
X_sel = pd.DataFrame(X_scaled, columns=FEATURE_COLS)[SELECTED].values

# 참가자 단위 분할
unique_p = np.asarray(feature_df['participant_id'].astype(str).unique().tolist())
train_p, test_p = train_test_split(unique_p, test_size=0.2, random_state=CONFIG["random_state"])
train_mask = feature_df['participant_id'].isin(train_p).values
test_mask = feature_df['participant_id'].isin(test_p).values

X_train, X_test = X_sel[train_mask], X_sel[test_mask]
y_train = feature_df.loc[train_mask, 'Arousal'].to_numpy(dtype=float)
y_test = feature_df.loc[test_mask, 'Arousal'].to_numpy(dtype=float)

print(f"학습: {X_train.shape[0]:,} 샘플 ({len(train_p)} 참가자)")
print(f"테스트: {X_test.shape[0]:,} 샘플 ({len(test_p)} 참가자)")

# GPU 설정
xgb_tree_method = 'hist'
xgb_device = 'cuda' if CONFIG["use_gpu"] else 'cpu'
lgb_device = 'gpu' if CONFIG["use_gpu"] else 'cpu'
cat_task_type = 'GPU' if CONFIG["use_gpu"] else 'CPU'

models = OrderedDict({
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=CONFIG["random_state"], n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, max_depth=6, random_state=CONFIG["random_state"], 
                                tree_method=xgb_tree_method, device=xgb_device, verbosity=0),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, max_depth=8, random_state=CONFIG["random_state"], 
                                  device=lgb_device, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=100, depth=6, random_state=CONFIG["random_state"], 
                                  task_type=cat_task_type, verbose=0, allow_writing_files=False),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, max_depth=12, random_state=CONFIG["random_state"], n_jobs=-1),
    'SVR': SVR(C=1.0, epsilon=0.1, gamma='scale')
})

cv = KFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=CONFIG["random_state"])
results_list = []

for name, model in models.items():
    print(f"  평가 중: {name}...", end=" ")
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1 if name != 'SVR' else 1)
    cv_rmse = np.sqrt(-cv_scores)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_r2 = r2_score(y_test, y_pred)
    results_list.append({'Model': name, 'CV_RMSE': cv_rmse.mean(), 'Test_RMSE': test_rmse, 'Test_R2': test_r2})
    print(f"CV RMSE: {cv_rmse.mean():.4f}")

results_df = pd.DataFrame(results_list).sort_values('CV_RMSE')
top_3 = results_df['Model'].head(3).tolist()

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71' if m in top_3 else '#95a5a6' for m in results_df['Model']]
axes[0].barh(results_df['Model'], results_df['CV_RMSE'], color=colors)
axes[0].set_xlabel('CV RMSE (lower = better)')
axes[0].set_title('Cross-Validation RMSE')
axes[0].invert_yaxis()
colors = ['#3498db' if m in top_3 else '#95a5a6' for m in results_df['Model']]
axes[1].barh(results_df['Model'], results_df['Test_R2'], color=colors)
axes[1].set_xlabel('Test R² (higher = better)')
axes[1].set_title('Test Set R² Score')
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\n🏆 상위 3개 모델: {top_3}")
display(results_df.round(4))


## 9. Optuna 하이퍼파라미터 튜닝

선정된 상위 3개 모델에 대해 Configuration에 정의된 범위 내에서 하이퍼파라미터를 최적화합니다.

In [ ]:
# ============================================================
# Optuna 튜닝
# ============================================================

def make_objective(model_name, X_tr, y_tr):
    ranges = CONFIG["optuna_ranges"][model_name]
    
    def objective(trial):
        if model_name == 'Random Forest':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', *ranges['n_estimators']),
                'max_depth': trial.suggest_int('max_depth', *ranges['max_depth']),
                'min_samples_split': trial.suggest_int('min_samples_split', *ranges['min_samples_split']),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', *ranges['min_samples_leaf'])
            }
            m = RandomForestRegressor(**params, random_state=CONFIG["random_state"], n_jobs=-1)
            
        elif model_name == 'Extra Trees':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', *ranges['n_estimators']),
                'max_depth': trial.suggest_int('max_depth', *ranges['max_depth']),
                'min_samples_split': trial.suggest_int('min_samples_split', *ranges['min_samples_split']),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', *ranges['min_samples_leaf'])
            }
            m = ExtraTreesRegressor(**params, random_state=CONFIG["random_state"], n_jobs=-1)
            
        elif model_name == 'XGBoost':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', *ranges['n_estimators']),
                'learning_rate': trial.suggest_float('learning_rate', *ranges['learning_rate'], log=True),
                'max_depth': trial.suggest_int('max_depth', *ranges['max_depth']),
                'subsample': trial.suggest_float('subsample', *ranges['subsample']),
                'colsample_bytree': trial.suggest_float('colsample_bytree', *ranges['colsample_bytree']),
                'reg_alpha': trial.suggest_float('reg_alpha', *ranges['reg_alpha'], log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', *ranges['reg_lambda'], log=True)
            }
            m = xgb.XGBRegressor(**params, random_state=CONFIG["random_state"], 
                                 tree_method=xgb_tree_method, device=xgb_device, verbosity=0)
            
        elif model_name == 'LightGBM':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', *ranges['n_estimators']),
                'learning_rate': trial.suggest_float('learning_rate', *ranges['learning_rate'], log=True),
                'num_leaves': trial.suggest_int('num_leaves', *ranges['num_leaves']),
                'max_depth': trial.suggest_int('max_depth', *ranges['max_depth']),
                'subsample': trial.suggest_float('subsample', *ranges['subsample']),
                'colsample_bytree': trial.suggest_float('colsample_bytree', *ranges['colsample_bytree']),
                'reg_alpha': trial.suggest_float('reg_alpha', *ranges['reg_alpha'], log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', *ranges['reg_lambda'], log=True)
            }
            m = lgb.LGBMRegressor(**params, random_state=CONFIG["random_state"], device=lgb_device, verbose=-1)
            
        elif model_name == 'CatBoost':
            params = {
                'iterations': trial.suggest_int('iterations', *ranges['iterations']),
                'learning_rate': trial.suggest_float('learning_rate', *ranges['learning_rate'], log=True),
                'depth': trial.suggest_int('depth', *ranges['depth']),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', *ranges['l2_leaf_reg'], log=True)
            }
            m = CatBoostRegressor(**params, random_state=CONFIG["random_state"], task_type=cat_task_type, verbose=0, allow_writing_files=False)
            
        else: # SVR
            params = {
                'C': trial.suggest_float('C', *ranges['C'], log=True),
                'epsilon': trial.suggest_float('epsilon', *ranges['epsilon'], log=True),
                'gamma': trial.suggest_float('gamma', *ranges['gamma'], log=True)
            }
            m = SVR(**params)
        
        scores = cross_val_score(m, X_tr, y_tr, cv=3, scoring='neg_mean_squared_error', n_jobs=-1 if model_name != 'SVR' else 1)
        return np.sqrt(-scores.mean())
    return objective

tuned_models = {}
tuning_results = []

for model_name in top_3:
    print(f"\n🔧 [{model_name}] 튜닝 중 ({CONFIG['n_trials']} trials)...")
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=CONFIG["random_state"]))
    study.optimize(make_objective(model_name, X_train, y_train), n_trials=CONFIG["n_trials"])
    
    bp = study.best_params
    
    # 최적 모델 재학습 (파라미터 주입 시 GPU 설정 유지)
    if model_name == 'Random Forest':
        bm = RandomForestRegressor(**bp, random_state=CONFIG["random_state"], n_jobs=-1)
    elif model_name == 'Extra Trees':
        bm = ExtraTreesRegressor(**bp, random_state=CONFIG["random_state"], n_jobs=-1)
    elif model_name == 'XGBoost':
        bm = xgb.XGBRegressor(**bp, random_state=CONFIG["random_state"], tree_method=xgb_tree_method, device=xgb_device, verbosity=0)
    elif model_name == 'LightGBM':
        bm = lgb.LGBMRegressor(**bp, random_state=CONFIG["random_state"], device=lgb_device, verbose=-1)
    elif model_name == 'CatBoost':
        bm = CatBoostRegressor(**bp, random_state=CONFIG["random_state"], task_type=cat_task_type, verbose=0, allow_writing_files=False)
    else:
        bm = SVR(**bp)
    
    bm.fit(X_train, y_train)
    yp = bm.predict(X_test)
    tuned_models[model_name] = {'model': bm, 'params': bp, 'study': study,
                                'rmse': np.sqrt(mean_squared_error(y_test, yp)),
                                'r2': r2_score(y_test, yp)}
    tuning_results.append({'Model': model_name, 'Best_CV_RMSE': study.best_value,
                          'Test_RMSE': tuned_models[model_name]['rmse'],
                          'Test_R2': tuned_models[model_name]['r2']})

tuning_df = pd.DataFrame(tuning_results).sort_values('Test_RMSE')
BEST_NAME = tuning_df.iloc[0]['Model']
BEST_PARAMS = tuned_models[BEST_NAME]['params']

print(f"\n🏆 최종 선정 모델: {BEST_NAME}")
print(f"   최적 하이퍼파라미터: {BEST_PARAMS}")
display(tuning_df.round(4))


## 10. 전체 타겟 변수에 대한 최종 모델 학습

In [ ]:
# ============================================================
# 5개 타겟 변수 학습
# ============================================================
final_models = {}
final_results = []

for target in TARGET_COLS:
    y_tr = feature_df.loc[train_mask, target].to_numpy(dtype=float)
    y_te = feature_df.loc[test_mask, target].to_numpy(dtype=float)
    
    if BEST_NAME == 'Random Forest':
        m = RandomForestRegressor(**BEST_PARAMS, random_state=CONFIG["random_state"], n_jobs=-1)
    elif BEST_NAME == 'Extra Trees':
        m = ExtraTreesRegressor(**BEST_PARAMS, random_state=CONFIG["random_state"], n_jobs=-1)
    elif BEST_NAME == 'XGBoost':
        m = xgb.XGBRegressor(**BEST_PARAMS, random_state=CONFIG["random_state"], tree_method=xgb_tree_method, device=xgb_device, verbosity=0)
    elif BEST_NAME == 'LightGBM':
        m = lgb.LGBMRegressor(**BEST_PARAMS, random_state=CONFIG["random_state"], device=lgb_device, verbose=-1)
    elif BEST_NAME == 'CatBoost':
        m = CatBoostRegressor(**BEST_PARAMS, random_state=CONFIG["random_state"], task_type=cat_task_type, verbose=0, allow_writing_files=False)
    else:
        m = SVR(**BEST_PARAMS)
    
    m.fit(X_train, y_tr)
    yp = m.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_te, yp))
    r2 = r2_score(y_te, yp)
    final_models[target] = m
    final_results.append({'Target': target, 'RMSE': rmse, 'R2': r2})
    print(f"  {target:20s} | RMSE: {rmse:.4f} | R²: {r2:.4f}")

final_df = pd.DataFrame(final_results)


## 11. 결과 시각화

In [ ]:
# ============================================================
# 시각화: Scatter Plot 및 잔차 분포
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes_flat = axes.flatten()

for i, target in enumerate(TARGET_COLS):
    ax = axes_flat[i]
    y_te = feature_df.loc[test_mask, target].to_numpy(dtype=float)
    yp = final_models[target].predict(X_test)
    ax.scatter(y_te, yp, alpha=0.3, s=10, c=plt.cm.tab10(i))
    lims = [min(y_te.min(), yp.min()), max(y_te.max(), yp.max())]
    ax.plot(lims, lims, 'r--', lw=2, label='Perfect Prediction')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{target} (R²={final_df.iloc[i]["R2"]:.3f})', fontweight='bold')
    ax.legend(fontsize=8)

axes_flat[5].set_visible(False)
plt.suptitle('실제값 vs 예측값 (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 12. 실시간 예측 파이프라인 (Real-time Inference)

학습된 모델을 실시간 바이오피드백 시스템에 적용하기 위한 파이프라인 클래스입니다.

In [ ]:
# ============================================================
# 실시간 예측 파이프라인 클래스
# ============================================================

class RealTimePPGEmotionPredictor:
    def __init__(self, models, scaler, all_feature_cols, selected_feature_cols,
                 sampling_rate=64, window_sec=30, step_sec=5):
        self.models = models
        self.scaler = scaler
        self.all_feature_cols = list(all_feature_cols)
        self.selected_feature_cols = list(selected_feature_cols)
        self.sampling_rate = sampling_rate
        self.window_size = window_sec * sampling_rate
        self.step_size = step_sec * sampling_rate
        
        self.buffer = np.zeros(self.window_size)
        self.buffer_idx = 0
        self.samples_since_last_pred = 0
        self.is_buffer_full = False
    
    def update(self, new_sample):
        self.buffer[self.buffer_idx] = new_sample
        self.buffer_idx = (self.buffer_idx + 1) % self.window_size
        self.samples_since_last_pred += 1
        
        if not self.is_buffer_full:
            if self.buffer_idx == 0:
                self.is_buffer_full = True
            else:
                return None
        
        if self.samples_since_last_pred >= self.step_size:
            self.samples_since_last_pred = 0
            return self._predict()
        return None
    
    def _predict(self):
        window = np.concatenate([
            self.buffer[self.buffer_idx:],
            self.buffer[:self.buffer_idx]
        ])
        features = extract_ppg_features(window, self.sampling_rate)
        feat_df = pd.DataFrame([features])
        for col in self.all_feature_cols:
            if col not in feat_df.columns:
                feat_df[col] = 0
        feat_df = feat_df[self.all_feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        feat_scaled_all = pd.DataFrame(
            self.scaler.transform(feat_df.values),
            columns=self.all_feature_cols,
        )
        feat_scaled = feat_scaled_all[self.selected_feature_cols].values
        
        predictions = {}
        for target, model in self.models.items():
            predictions[target] = round(float(model.predict(feat_scaled)[0]), 3)
        return predictions

predictor = RealTimePPGEmotionPredictor(
    models=final_models,
    scaler=scaler,
    all_feature_cols=FEATURE_COLS,
    selected_feature_cols=SELECTED,
    sampling_rate=CONFIG["sampling_rate"],
    window_sec=CONFIG["window_sec"],
    step_sec=CONFIG["step_sec"],
)

for sample in all_data[0][0][:predictor.window_size]:
    predictor.update(sample)

print("실시간 예측 파이프라인 준비 완료")
print("샘플 예측:", predictor._predict())
